# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [4]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/AI/Clase 6/TU_CARPETA")
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

assert PROJECT_DIR.exists(), f"No encuentro la carpeta del proyecto: {PROJECT_DIR}"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print("PROJECT_DIR:", PROJECT_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/AI/Clase 6/TU_CARPETA


## 1. Carga de artefactos

In [5]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [6]:
feature_columns = contract["features"]

# Para árboles: solo imputación por mediana (no necesitan escalado)
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), feature_columns),
    ],
    remainder="drop",
)

# Para Regresión Logística y KNN: imputación por mediana + estandarización
scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), feature_columns),
    ],
    remainder="drop",
)

print("Preprocesadores creados:", type(tree_preprocessor).__name__, type(scaled_preprocessor).__name__)

Preprocesadores creados: ColumnTransformer ColumnTransformer


**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Porque si el StandardScaler se ajusta antes de la validación cruzada, la media y la desviación estándar se calculan con todos los datos, incluidos los de los folds de validación, lo que filtra información (data leakage) y vuelve optimista el score de CV; dentro del pipeline se reajusta en cada fold usando solo sus datos de entrenamiento.

## 3. Modelos candidatos

In [7]:
models = {
    "DecisionTree": Pipeline([
        ("preprocess", clone(tree_preprocessor)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "LogisticRegression": Pipeline([
        ("preprocess", clone(scaled_preprocessor)),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "KNN": Pipeline([
        ("preprocess", clone(scaled_preprocessor)),
        ("model", KNeighborsClassifier()),
    ]),
}

for name, pipe in models.items():
    print(f"{name}: {[step for step, _ in pipe.steps]}")

DecisionTree: ['preprocess', 'model']
LogisticRegression: ['preprocess', 'model']
KNN: ['preprocess', 'model']


## 4. Comparación inicial con validación cruzada

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipe in models.items():
    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=-1,
    )
    rows.append({
        "model": name,
        "train_accuracy_mean": scores["train_score"].mean(),
        "val_accuracy_mean": scores["test_score"].mean(),
        "val_accuracy_std": scores["test_score"].std(),
    })

initial_comparison = (
    pd.DataFrame(rows)
    .assign(gap=lambda d: d["train_accuracy_mean"] - d["val_accuracy_mean"])
    .sort_values("val_accuracy_mean", ascending=False)
    .reset_index(drop=True)
)

initial_comparison.round(4)

,model,train_accuracy_mean,val_accuracy_mean,val_accuracy_std,gap
0,LogisticRegression,1.0000,0.9791,0.0277,0.0209
1,KNN,0.9771,0.9507,0.0353,0.0264
2,DecisionTree,1.0000,0.9022,0.0586,0.0978


## 5. Optimización de hiperparámetros

In [9]:
# Espacios de búsqueda pequeños (el prefijo "model__" apunta al paso "model" del pipeline)
param_grids = {
    "DecisionTree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 2, 4, 8],
        "model__criterion": ["gini", "entropy"],
    },
    "LogisticRegression": {
        "model__C": [0.01, 0.1, 1, 10, 100],
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11, 15],
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2],
    },
}

searches = {}
rows = []
for name, pipe in models.items():
    search = GridSearchCV(
        estimator=clone(pipe),
        param_grid=param_grids[name],
        scoring="accuracy",
        cv=cv,
        return_train_score=True,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search

    best = search.best_index_
    rows.append({
        "model": name,
        "best_cv_accuracy": search.best_score_,
        "cv_std": search.cv_results_["std_test_score"][best],
        "train_accuracy": search.cv_results_["mean_train_score"][best],
        "best_params": search.best_params_,
    })

tuned_comparison = (
    pd.DataFrame(rows)
    .assign(gap=lambda d: d["train_accuracy"] - d["best_cv_accuracy"])
    .sort_values("best_cv_accuracy", ascending=False)
    .reset_index(drop=True)
)

tuned_comparison

,model,best_cv_accuracy,cv_std,train_accuracy,best_params,gap
0,LogisticRegression,0.979064,0.027713,1.000000,{'model__C': 1},0.020936
1,KNN,0.978818,0.017301,0.980624,"{'model__n_neighbors': 7, 'model__p': 1, 'mode...",0.001806
2,DecisionTree,0.902463,0.076225,0.985934,"{'model__criterion': 'gini', 'model__max_depth...",0.083471


## 6. Selección y persistencia del campeón

In [10]:
import sklearn

# --- Selección: mayor best_score_ de CV ---
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion_search = searches[champion_name]
champion = champion_search.best_estimator_  # pipeline completo, reentrenado con todo train

print("Campeón:", champion_name)
print("Mejores hiperparámetros:", champion_search.best_params_)
print(f"CV accuracy: {champion_search.best_score_:.4f}")

# --- Persistencia del modelo ---
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

# --- cv_results.csv: resultados de todas las búsquedas ---
cv_frames = []
for name, search in searches.items():
    res = pd.DataFrame(search.cv_results_)
    res["params"] = res["params"].astype(str)  # los dicts no se guardan bien en CSV
    res.insert(0, "model", name)
    cv_frames.append(res)

cv_results_df = pd.concat(cv_frames, ignore_index=True)
cv_results_df.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

# --- training_metadata.json ---
def to_jsonable(obj):
    return obj.item() if hasattr(obj, "item") else str(obj)

best_idx = champion_search.best_index_
training_metadata = {
    "champion": champion_name,
    "best_params": champion_search.best_params_,
    "cv_accuracy_mean": float(champion_search.best_score_),
    "cv_accuracy_std": float(champion_search.cv_results_["std_test_score"][best_idx]),
    "train_accuracy_mean": float(champion_search.cv_results_["mean_train_score"][best_idx]),
    "selection_metric": "accuracy",
    "cv": {"type": "StratifiedKFold", "n_splits": 5, "shuffle": True, "random_state": RANDOM_STATE},
    "random_state": RANDOM_STATE,
    "target": contract["target"],
    "features": contract["features"],
    "n_train": int(len(X_train)),
    "candidates": {
        name: {
            "best_params": s.best_params_,
            "cv_accuracy_mean": float(s.best_score_),
        }
        for name, s in searches.items()
    },
    "sklearn_version": sklearn.__version__,
}

with open(ARTIFACTS_DIR / "training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, indent=2, ensure_ascii=False, default=to_jsonable)

# --- Sanidad: recargar el modelo y predecir con él (solo train) ---
reloaded = joblib.load(ARTIFACTS_DIR / "champion_model.joblib")
print("Predicciones de prueba (primeras 5 filas de train):", reloaded.predict(X_train.head()))
print("Reales:", y_train.head().to_list())

Campeón: LogisticRegression
Mejores hiperparámetros: {'model__C': 1}
CV accuracy: 0.9791
Predicciones de prueba (primeras 5 filas de train): [0 0 0 0 2]
Reales: [0, 0, 0, 0, 2]


In [11]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Explica la decisión considerando desempeño, variabilidad, sobreajuste, complejidad e interpretabilidad.

Elegí la Regresión Logística (C = 1) por tener el mayor accuracy medio en validación cruzada (0.979), con una brecha train–validación pequeña (0.021), lo que indica poco sobreajuste. KNN (n_neighbors = 7) quedó prácticamente empatado (0.979) y es incluso más estable entre folds (std 0.017 frente a 0.028); la diferencia de 0.0002 no es significativa, así que el desempate lo hacen la complejidad y la interpretabilidad. La Regresión Logística es un modelo simple con coeficientes interpretables por variable, y su predicción solo requiere guardar un vector de pesos, mientras que KNN debe almacenar todo el conjunto de entrenamiento y es más sensible a la escala y a la distancia. El árbol de decisión fue claramente peor (0.902), con la mayor variabilidad (std 0.076) y el mayor sobreajuste (brecha 0.083). La decisión se tomó solo con CV sobre entrenamiento, sin usar el test.

## Checklist

- [ ] Todo el procesamiento está dentro de pipelines.
- [ ] Comparé tres familias de modelos.
- [ ] Optimicé hiperparámetros con CV.
- [ ] Elegí el campeón sin usar test.
- [ ] Guardé el pipeline completo y sus metadatos.